In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# ============================================================
# Generate Synthetic Ground Truth Image
# ============================================================

N = 32

x_grid = np.linspace(-1, 1, N)
y_grid = np.linspace(-1, 1, N)

X, Y = np.meshgrid(x_grid, y_grid)

# Smooth synthetic image
ground_truth = np.exp(-(X**2 + Y**2) * 5)

ground_truth = torch.tensor(
    ground_truth,
    dtype=torch.float32
).unsqueeze(0).unsqueeze(0)

# ============================================================
# Add Gaussian Noise
# ============================================================

noise_level = 0.05

observed = ground_truth + noise_level * torch.randn_like(ground_truth)

# ============================================================
# DIP-style CNN
# ============================================================

class DIPNet(nn.Module):

    def __init__(self, width=64):

        super(DIPNet, self).__init__()

        self.net = nn.Sequential(

            nn.Conv2d(1, width, kernel_size=3, padding=1),
            nn.Tanh(),

            nn.Conv2d(width, width, kernel_size=3, padding=1),
            nn.Tanh(),

            nn.Conv2d(width, 1, kernel_size=3, padding=1)

        )

    def forward(self, x):

        return self.net(x)

# ============================================================
# Model Initialization
# ============================================================

model = DIPNet(width=64)

# Fixed random DIP input
z = torch.randn(1, 1, N, N)

# ============================================================
# Optimization Setup
# ============================================================

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=1e-2)

num_iterations = 1000

loss_history = []

# ============================================================
# DIP Optimization
# ============================================================

for iteration in range(num_iterations):

    optimizer.zero_grad()

    output = model(z)

    loss = criterion(output, observed)

    loss.backward()

    optimizer.step()

    loss_history.append(loss.item())

    if iteration % 100 == 0:

        print(f"Iteration {iteration:4d} | Loss = {loss.item():.6e}")

# ============================================================
# Plot Loss Decay
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(loss_history)

plt.xlabel("Iteration")
plt.ylabel("Loss")

plt.title("DIP Optimization Loss Decay")

plt.grid(True)

plt.show()

# ============================================================
# Semi-log Plot
# ============================================================

plt.figure(figsize=(8,5))

plt.semilogy(loss_history)

plt.xlabel("Iteration")
plt.ylabel("Loss (log scale)")

plt.title("Semi-log Plot of DIP Optimization Loss")

plt.grid(True)

plt.show()

# ============================================================
# Reconstructed Image
# ============================================================

reconstructed = output.detach().cpu().numpy()[0,0]

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(ground_truth[0,0], cmap='gray')
plt.title("Ground Truth")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(observed[0,0], cmap='gray')
plt.title("Observed")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(reconstructed, cmap='gray')
plt.title("DIP Reconstruction")
plt.axis('off')

plt.show()

# ============================================================
# Estimate Exponential Convergence Rate
# ============================================================

loss_array = np.array(loss_history)

# Avoid log(0)
loss_array = np.maximum(loss_array, 1e-12)

log_loss = np.log(loss_array)

# Linear fit
iterations = np.arange(num_iterations)

coef = np.polyfit(iterations, log_loss, 1)

rate = -coef[0]

print("\n================================================")
print("Estimated Exponential Convergence Rate")
print("================================================")
print(f"Estimated rate = {rate:.6e}")
print("================================================")

Iteration    0 | Loss = 8.539785e-02
Iteration  100 | Loss = 3.282833e-03
Iteration  200 | Loss = 5.053770e-04
Iteration  300 | Loss = 1.116993e-04
Iteration  400 | Loss = 2.576633e-05
